# backward-on-scalar-loss — ex1: reduce per-sample loss to a scalar before backward

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backward-on-scalar-loss`. Running the final beacon cell reports progress against the `PyTorch: backward()` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: backward()` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-on-scalar-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-on-scalar-loss"
DD_SUBTOPIC = "PyTorch: backward()"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `loss.backward()` on a scalar — quick refresher

`tensor.backward()` populates `.grad` on every leaf tensor with `requires_grad=True` that contributed to `tensor`. PyTorch requires the tensor to be a SCALAR (0-D / single-element) unless you pass a gradient-output tensor. Hence the universal training loop reduces the per-sample loss to a single number first — typically `.mean()` or `.sum()` — and only then calls `.backward()`.

**Gradient accumulates.** Each call to `.backward()` ADDS into existing `.grad`. That's why the loop ends with `optimizer.zero_grad()` — to reset accumulation between batches.

**Backward requires a graph.** If you wrap the forward in `t.no_grad()` or call `.detach()`, the resulting tensor has no `grad_fn` and `.backward()` will raise.

### Exercise 1 — reduce per-sample loss to a scalar before backward

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply scalar reduction (`.mean()`) to a per-sample loss vector and call `.backward()` on the scalar so that the leaf parameter's `.grad` is populated correctly.
> Keywords: backward, scalar-reduction, mean-vs-sum
> ```

**KCs targeted:** `backward-requires-scalar-loss`, `backward-populates-leaf-grad`

Implement `ex1_backward_mean(w, x, y)`. A common ARENA pattern: compute a per-sample squared-error loss, reduce to a scalar with `.mean()`, and call `.backward()`.

1. Compute `per_sample_loss = (w * x - y) ** 2`. This is a vector with the same shape as `x`.
2. Reduce to a scalar with `.mean()` → `scalar_loss`.
3. Call `scalar_loss.backward()`.
4. Return the tuple `(scalar_loss, per_sample_loss)`. The test verifies (a) the scalar is correct, (b) `w.grad` is correct, (c) per-sample loss has the right shape.

Inputs:
- `w`: a leaf tensor with `requires_grad=True`, shape `(1,)`.
- `x, y`: 1-D float tensors of equal length.

**Common trap.** Calling `.backward()` directly on the per-sample loss tensor (without reducing) raises `RuntimeError: grad can be implicitly created only for scalar outputs`. The scalar reduction is mandatory.

In [ ]:
def ex1_backward_mean(w, x, y):
    per_sample_loss = (w * x - y) ** 2
    scalar_loss = per_sample_loss.mean()
    scalar_loss.backward()
    return scalar_loss, per_sample_loss


<details><summary>Solution</summary>

```python
def ex1_backward_mean(w, x, y):
    per_sample_loss = (w * x - y) ** 2
    scalar_loss = per_sample_loss.mean()
    scalar_loss.backward()
    return scalar_loss, per_sample_loss
```

**Mean vs sum changes the gradient magnitude.** `.mean()` divides by N, so the gradient is the average per-sample gradient. `.sum()` is N× larger, which acts like an effective lr of `lr * N`. Karpathy's micrograd/nano-GPT both use `.mean()`; that's the default convention for `nn.MSELoss` and `nn.CrossEntropyLoss` too (`reduction='mean'`).

**The 'implicit scalar' check is fundamental.** Internally PyTorch's `.backward()` creates a 1.0 gradient at the output of the graph and propagates from there. For a scalar that's unambiguous. For a vector PyTorch refuses to guess; you'd have to pass `gradient=t.ones_like(vec)` explicitly, which is advanced. Production training loops never do this — they always reduce.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()